# Поступашки: воспроизводимая проверка MVP

Этот notebook дополняет исходный Colab коллеги. Исторические оплаты, наблюдаемые посты и синтетические касания читаются раздельно. Совпадение курса и даты не устанавливает реальный источник клиента. Запускать из корня репозитория или папки notebooks. Для Google Colab сначала загрузите data/ и docs/.


In [1]:
from pathlib import Path
import json, hashlib
from collections import Counter
ROOT = Path.cwd() if (Path.cwd() / 'data/seed.json').exists() else Path.cwd().parent
seed = json.loads((ROOT / 'data/seed.json').read_text())
manifest = json.loads((ROOT / 'data/manifest.json').read_text())


In [2]:
payments = seed['payments']
historical = [p for p in payments if p['mode'] == 'historical']
assert len(historical) == 795
assert len({p['userId'] for p in historical}) == 606
assert len(seed['courses']) == 18
assert all(p['userId'].startswith('hist_') for p in historical)
assert not {p['userId'] for p in historical} & {e['userId'] for e in seed['events']}
print('Строк оплат:', len(historical), 'Уникальных ID:', 606, 'Курсов:', 18)
print('Календарь по типу:', Counter(p['mode'] for p in seed['placements']))


Строк оплат: 795 Уникальных ID: 606 Курсов: 18
Календарь по типу: Counter({'historical': 62, 'demo': 13})


In [3]:
counts = Counter(p['course'] for p in historical)
sums = Counter()
for p in historical: sums[p['course']] += p['amount']
for course, rows in counts.most_common():
    print(course, rows, round(sums[course], 2))


Аналитика про 93 740941.5
Аналитика старт 91 641616.5
AI агенты 91 711673.0
ML про 90 715110.0
Алгоритмы старт 88 529851.68
ML старт 80 523278.33
Backend про 55 438342.5
Backend старт 39 263396.66
Алгоритмы про 35 268409.0
Алгоритмы 30 209482.5
К ВУЗу 26 180250.0
Мат анализ 20 191382.5
Линейная алгебра 19 180665.0
Теория вероятностей 13 132107.5
Data Science 9 69295.0
АВ тестам 8 49610.0
Data Engenering 4 24845.0
Дискретка 4 34415.0


## Разметка и ограничения

В календаре 62 проверенных публикации из 182 наблюдаемых. Явные курсы учитываются по умолчанию; кандидаты по линейке требуют включения допущения. Расходы на историческую рекламу неизвестны. Демо-расходы и оплаты вымышлены.


In [4]:
posts = [p for p in seed['placements'] if p['mode'] == 'historical']
assert len(posts) == 62
assert all(p['cost'] is None for p in posts)
print(Counter(p['match'] for p in posts))
print('Связанные оригиналы:', sum(bool(p['originUrl']) for p in posts))


Counter({'family_candidate': 35, 'explicit': 16, 'none': 10, 'unknown': 1})
Связанные оригиналы: 12


## Результаты проверок движка

Результаты получены запуском node scripts/run_tests.mjs. Окна и часовой пояс являются допущениями. Суммы распределений проверяются для всех моделей, включая неизвестный источник.


In [5]:
report = json.loads((ROOT / 'docs/validation-results.json').read_text())
print(json.dumps(report, ensure_ascii=False, indent=2))
api = json.loads((ROOT / 'docs/api-validation-results.json').read_text())
assert api['passed']
print('Проверки API:', api['count'], api['environment'])


{
  "historical": [
    {
      "clock": 0,
      "windows": [
        {
          "windowDays": 3,
          "matched": 143,
          "unknownAmount": 4938574.49
        },
        {
          "windowDays": 5,
          "matched": 176,
          "unknownAmount": 4712684.49
        },
        {
          "windowDays": 7,
          "matched": 217,
          "unknownAmount": 4399064.49
        }
      ]
    },
    {
      "clock": 3,
      "windows": [
        {
          "windowDays": 3,
          "matched": 140,
          "unknownAmount": 4959064.49
        },
        {
          "windowDays": 5,
          "matched": 165,
          "unknownAmount": 4783589.49
        },
        {
          "windowDays": 7,
          "matched": 211,
          "unknownAmount": 4429944.49
        }
      ]
    }
  ],
  "demoModels": [
    {
      "model": "first",
      "results": [
        {
          "campaign": "Набор: аналитика",
          "course": "Аналитика про",
          "netRevenue": 64950,
   

In [6]:
demo_posts = [p for p in seed["placements"] if p["mode"] == "demo"]
assert len(demo_posts) == 13
assert all(p.get("linkIssuedAt") for p in demo_posts)
assert len(seed["leads"]) == 48
assert len(seed["events"]) == 136
assert len([p for p in payments if p["mode"] == "demo"]) == 16
assert len({e["userId"] for e in seed["events"]}) == 60
print("Демо v2: 13 размещений, 60 посетителей, 48 заявок, 136 событий, 16 оплат")


Демо v2: 13 размещений, 60 посетителей, 48 заявок, 136 событий, 16 оплат


## Следующий проверяемый запуск

Создать размещение, выпустить ссылку в календаре и вставить в пост. Сохранить адрес публикации, собрать заявки и подтверждённые оплаты. Заполнить расходы кампании и экономику курса, дождаться зрелого результата. Имена команды добавляются позднее.
